## Cell 1 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q pmdarima

import pandas as pd
import numpy as np

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.0 MB/s eta 0:00:00


# Cell 2 (SARIMA version) - Verify imports


In [2]:
import statsmodels.api as sm
print('statsmodels:', sm.__version__)

statsmodels: 0.14.6


# Cell 3 - Load processed data


In [3]:
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


# Cell 4 - WMAE metric


In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

## Cell 5 — Sample Selection (Same 15 Series as ARIMA/Prophet, seed=42)

In [5]:

SAMPLE_SIZE = 15
VAL_LEN = 39
MIN_LEN = 52 + VAL_LEN + 10

series_totals = (
    train.groupby(['Store','Dept'])['Weekly_Sales']
    .sum()
    .sort_values(ascending=False)
)
np.random.seed(42)
all_keys = series_totals.index.tolist()
sample_idx = np.random.choice(len(all_keys), size=SAMPLE_SIZE, replace=False)
sample_keys = [all_keys[i] for i in sample_idx]

series_data = {}
for store, dept in sample_keys:
    sub = train[(train['Store']==store) & (train['Dept']==dept)].sort_values('Date')
    if len(sub) < MIN_LEN:
        continue
    sub = sub.set_index('Date').asfreq('W-FRI')
    sub['Weekly_Sales'] = sub['Weekly_Sales'].interpolate().bfill().ffill()
    sub['IsHoliday'] = sub['IsHoliday'].astype(float).fillna(0.0).astype(bool)
    series_data[(store, dept)] = {
        'values': sub['Weekly_Sales'].values,
        'is_holiday': sub['IsHoliday'].values
    }

print('Series selected for ARIMA:', len(series_data))

Series selected for ARIMA: 13


## Cell 6 — Chronological 39-Week Holdout Split

In [6]:
fit_raw, val_raw, val_holiday = {}, {}, {}
for key, d in series_data.items():
    fit_raw[key] = d['values'][:-VAL_LEN]
    val_raw[key] = d['values'][-VAL_LEN:]
    val_holiday[key] = d['is_holiday'][-VAL_LEN:]

## Cell 7 — Fit SARIMA(1,1,1)(1,1,1,52) Per Series

In [7]:
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_models = {}
all_true, all_pred, all_holiday = [], [], []

for key, fseries in fit_raw.items():
    model = SARIMAX(
        fseries,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, 52),
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    fitted = model.fit(disp=False, maxiter=50)
    sarima_models[key] = fitted

    pred = fitted.forecast(steps=VAL_LEN)
    all_true.extend(val_raw[key])
    all_pred.extend(pred)
    all_holiday.extend(val_holiday[key])

    print(f"{key}: fit done, AIC={fitted.aic:.1f}")

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_holiday = np.array(all_holiday)
print('Done. Total validation points:', len(all_true))

(41, 13): fit done, AIC=10.0
(26, 28): fit done, AIC=10.0
(32, 67): fit done, AIC=10.0
(28, 30): fit done, AIC=10.0
(18, 27): fit done, AIC=10.0
(15, 32): fit done, AIC=10.0
(44, 92): fit done, AIC=10.0
(18, 36): fit done, AIC=10.0
(15, 6): fit done, AIC=10.0
(8, 29): fit done, AIC=10.0
(16, 52): fit done, AIC=10.0
(16, 79): fit done, AIC=10.0
(24, 14): fit done, AIC=10.0
Done. Total validation points: 507


## Cell 8 — Evaluate

In [8]:
sarima_wmae = wmae(all_true, all_pred, all_holiday)
print('SARIMA WMAE:', sarima_wmae)
print('Holiday WMAE:    ', wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday]))
print('Non-holiday WMAE:', wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday]))

SARIMA WMAE: 1096.0113680027391
Holiday WMAE:     1010.4401812741613
Non-holiday WMAE: 1119.1387157672198


## Cell 9 — Save Models Locally

In [9]:
import os
import joblib

os.makedirs('models', exist_ok=True)
joblib.dump(sarima_models, 'models/sarima_best.pkl')
print('Saved.')

Saved.


## Cell 10 — 52-Week Seasonal Baseline (Same Sample as SARIMA)

In [10]:
baseline_pred_list = []
for key in sarima_models.keys():
    vals = series_data[key]['values']
    fit_vals = vals[:-VAL_LEN]
    if len(fit_vals) >= VAL_LEN+52:
        lookback = fit_vals[-VAL_LEN-52:-52]
    else:
        lookback = np.full(VAL_LEN, fit_vals.mean())
    baseline_pred_list.extend(lookback[:VAL_LEN])

baseline_pred_arr = np.array(baseline_pred_list)
baseline_score = wmae(all_true, baseline_pred_arr, all_holiday)
print('52-week seasonal baseline WMAE:', baseline_score)
print('SARIMA WMAE:                    ', sarima_wmae)

52-week seasonal baseline WMAE: 2549.7130769230766
SARIMA WMAE:                     1096.0113680027391


## Cell 11 — MLflow / DagsHub Setup

In [11]:
!pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='skupr23', repo_name='ml_final', mlflow=True)

import mlflow
import mlflow.sklearn

EXPERIMENT_NAME = 'SARIMA_Training'
REGISTERED_MODEL_NAME = 'WalmartSalesForecast'

mlflow.set_experiment(EXPERIMENT_NAME)
if mlflow.active_run() is not None:
    mlflow.end_run()

print('Experiment:', EXPERIMENT_NAME)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=4ec6a90b-97c2-4cab-8f2d-23ef5ee6c911&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3100bb13b2af0c310c7cbb036c7da798bfb30d06c034c08f20070fb78db62636




Accessing as ggzob23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

2026/07/12 11:26:07 INFO mlflow.tracking.fluent: Experiment with name 'SARIMA_Training' does not exist. Creating a new experiment.


Experiment: SARIMA_Training


## Cell 12 — Run 1: Cleaning

In [12]:
with mlflow.start_run(run_name='SARIMA_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'freq': 'W-FRI',
        'interpolation': 'linear + bfill/ffill',
        'dataset_scope': f'sample ({SAMPLE_SIZE} series, seed=42, same population as ARIMA/Prophet sample notebooks)',
        'min_series_length': MIN_LEN,
        'sampling_seed': 42,
    })
    mlflow.log_metrics({
        'total_store_dept_combinations': len(all_keys),
        'sample_size': SAMPLE_SIZE,
        'series_kept': len(series_data),
        'val_weeks': VAL_LEN,
    })

🏃 View run SARIMA_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12/runs/1eae95ac6f6140ea81f901793a35dbb7
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12


## Cell 13 — Run 2: Feature Selection Reasoning

In [13]:
with mlflow.start_run(run_name='SARIMA_Feature_Selection'):
    mlflow.set_tag('stage', 'feature_selection')
    mlflow.log_params({
        'selection_strategy': 'univariate only - no exogenous regressors',
        'model_input': 'Weekly_Sales history only',
        'reason': 'SARIMA adds a seasonal term over plain ARIMA but still has no '
                   'built-in mechanism for external regressors. Per assignment '
                   'instructions, ARIMA/SARIMA get light theoretical treatment '
                   'rather than heavy engineering.',
    })

🏃 View run SARIMA_Feature_Selection at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12/runs/d1d4db920ae149459127666ad1b00235
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12


## Cell 14 — Run 3: Baseline

In [14]:
with mlflow.start_run(run_name='SARIMA_Baseline'):
    mlflow.set_tag('stage', 'baseline')
    mlflow.log_param('strategy', f'52-week seasonal naive, same {SAMPLE_SIZE}-series sample as SARIMA')
    mlflow.log_metric('val_wmae', baseline_score)

🏃 View run SARIMA_Baseline at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12/runs/4f2bfa4b2cb44f04b87e377137a952a7
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12


## Cell 15 — Run 4: Validation

In [15]:
with mlflow.start_run(run_name='SARIMA_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.set_tag('population', f'sample ({SAMPLE_SIZE} series) - NOT directly comparable to full-population models like XGBoost/LightGBM/Prophet-full')
    mlflow.log_params({
        'scheme': 'chronological holdout, 39-week convention, sample of series',
        'val_weeks': VAL_LEN,
        'order': '(1, 1, 1)',
        'seasonal_order': '(1, 1, 1, 52)',
        'search_performed': False,
        'reason_no_search': 'Assignment instructs minimal training time for SARIMA; '
                             'a fixed order across a representative sample keeps this '
                             'feasible without an hours-long per-series seasonal search.',
        'sample_size': SAMPLE_SIZE,
        'sampling_seed': 42,
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_score,
        'sarima_wmae': sarima_wmae,
        'improvement_over_baseline': baseline_score - sarima_wmae,
    })

🏃 View run SARIMA_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12/runs/671e7406931a448ba06abdee390ad6fd
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12


## Cell 16 — Run 5: Final Model

In [16]:
with mlflow.start_run(run_name='SARIMA_Final_Model') as final_run:
    mlflow.set_tags({
        'stage': 'final_model',
        'algorithm': 'sarima',
        'dataset_scope': f'sample_{SAMPLE_SIZE}_series',
    })
    mlflow.log_metrics({
        'final_wmae': sarima_wmae,
        'baseline_wmae': baseline_score,
    })
    mlflow.log_artifact('models/sarima_best.pkl', artifact_path='estimator')

    FINAL_RUN_ID = final_run.info.run_id

print('SARIMA final run:', FINAL_RUN_ID)

🏃 View run SARIMA_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12/runs/c42215bec39245babd2d2ee6f46c38e0
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/12
SARIMA final run: c42215bec39245babd2d2ee6f46c38e0


## Cell 17 — Register (Only If SARIMA Wins Overall)

In [17]:
REGISTER_AS_GLOBAL_BEST = False

if REGISTER_AS_GLOBAL_BEST:
    registered_version = mlflow.register_model(
        model_uri=f'runs:/{FINAL_RUN_ID}/estimator',
        name=REGISTERED_MODEL_NAME,
    )
    print('Registered model:', REGISTERED_MODEL_NAME, '| version:', registered_version.version)
else:
    print('Pipeline logged and saved, not registered.')

Pipeline logged and saved, not registered.
